In [1]:
"""
=============================================================
MODEL 1 — XGBOOST BASELINE
=============================================================
Multi-output regression using separate XGBoost per target.
Runs on CPU. No GPU needed.

Requirements:
  pip install xgboost scikit-learn pandas numpy

Usage:
  python train_model1_xgboost.py
=============================================================
"""

import os
import time
import json
import pickle
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_DIR   = './ml_outputs_enriched'
OUTPUT_DIR = './model_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_COLS = [
    'TARGET_threshold_hours',
    'TARGET_threshold_inactive_min',
    'TARGET_threshold_fluctuation',
    'TARGET_each_hour_fluctuation',
]
ID_COLS = ['Cell', 'Alarm_Raised_Date', 'Start_Hour']

# ── Load data ─────────────────────────────────────────────────
print("Loading data...")
X_train_df = pd.read_csv(f'{DATA_DIR}/X_train.csv')
X_test_df  = pd.read_csv(f'{DATA_DIR}/X_test.csv')
y_train_df = pd.read_csv(f'{DATA_DIR}/y_train.csv')
y_test_df  = pd.read_csv(f'{DATA_DIR}/y_test.csv')

FEAT_COLS = [c for c in X_train_df.columns if c not in ID_COLS]

# Impute NaN
for col in FEAT_COLS:
    if X_train_df[col].isnull().any():
        m = X_train_df[col].median()
        X_train_df[col] = X_train_df[col].fillna(m)
        X_test_df[col]  = X_test_df[col].fillna(m)

X_train = X_train_df[FEAT_COLS].values.astype(np.float32)
X_test  = X_test_df[FEAT_COLS].values.astype(np.float32)
y_train = y_train_df[TARGET_COLS].values.astype(np.float32)
y_test  = y_test_df[TARGET_COLS].values.astype(np.float32)

print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"  X_test:  {X_test.shape},  y_test:  {y_test.shape}")
print(f"  Features: {len(FEAT_COLS)}")

# ── XGBoost params per target ─────────────────────────────────
# Each target has slightly different characteristics:
# - threshold_hours: integer 2-8, use huber loss
# - threshold_inactive_min: continuous, heavy right skew, use huber
# - threshold_fluctuation: mostly 1, occasional spike, use huber
# - each_hour_fluctuation: same as above

PARAMS = {
    'TARGET_threshold_hours': dict(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=20,
        reg_alpha=0.1,
        reg_lambda=1.0,
        objective='reg:pseudohubererror',
        huber_slope=1.0,
        random_state=42,
        n_jobs=-1,
    ),
    'TARGET_threshold_inactive_min': dict(
        n_estimators=500,
        max_depth=7,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.7,
        min_child_weight=10,
        reg_alpha=0.1,
        reg_lambda=1.0,
        objective='reg:pseudohubererror',
        huber_slope=2.0,
        random_state=42,
        n_jobs=-1,
    ),
    'TARGET_threshold_fluctuation': dict(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=30,
        reg_alpha=0.5,
        reg_lambda=1.0,
        objective='reg:pseudohubererror',
        huber_slope=0.5,
        random_state=42,
        n_jobs=-1,
    ),
    'TARGET_each_hour_fluctuation': dict(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=30,
        reg_alpha=0.5,
        reg_lambda=1.0,
        objective='reg:pseudohubererror',
        huber_slope=0.5,
        random_state=42,
        n_jobs=-1,
    ),
}

CLIP_RANGES = {
    'TARGET_threshold_hours':        (2, 8),
    'TARGET_threshold_inactive_min': (5, 200),
    'TARGET_threshold_fluctuation':  (1, 7),
    'TARGET_each_hour_fluctuation':  (1, 7),
}

# ── Train ─────────────────────────────────────────────────────
print("\nTraining XGBoost models...")
print(f"{'Target':<35} {'Train Time':>12} {'MAE':>8} {'RMSE':>8} {'R²':>8}")
print("-" * 75)

models     = {}
results    = {}
all_preds  = np.zeros_like(y_test)
t_total    = time.time()

for i, target in enumerate(TARGET_COLS):
    t0  = time.time()
    yt  = y_train[:, i]
    yt_test = y_test[:, i]

    model = XGBRegressor(**PARAMS[target])
    model.fit(
        X_train, yt,
        eval_set=[(X_test, yt_test)],
        verbose=False,
    )

    pred = model.predict(X_test)
    lo, hi = CLIP_RANGES[target]
    pred = np.clip(pred, lo, hi)
    all_preds[:, i] = pred

    mae  = mean_absolute_error(yt_test, pred)
    rmse = np.sqrt(mean_squared_error(yt_test, pred))
    r2   = r2_score(yt_test, pred)
    elapsed = time.time() - t0

    short = target.replace('TARGET_', '')
    print(f"  {short:<33} {elapsed:>10.1f}s {mae:>8.4f} {rmse:>8.4f} {r2:>8.4f}")

    models[target]  = model
    results[target] = {'MAE': round(mae,4), 'RMSE': round(rmse,4), 'R2': round(r2,4)}

total_time = time.time() - t_total
print("-" * 75)
avg_mae  = np.mean([v['MAE']  for v in results.values()])
avg_rmse = np.mean([v['RMSE'] for v in results.values()])
avg_r2   = np.mean([v['R2']   for v in results.values()])
print(f"  {'AVERAGE':<33} {total_time:>10.1f}s "
      f"{avg_mae:>8.4f} {avg_rmse:>8.4f} {avg_r2:>8.4f}")

# ── Feature importances ───────────────────────────────────────
print("\nTop 15 features (average importance across targets):")
importances = np.mean([
    models[t].feature_importances_ for t in TARGET_COLS
], axis=0)
top_idx = np.argsort(importances)[::-1][:15]
for rank, idx in enumerate(top_idx, 1):
    print(f"  {rank:2d}. {FEAT_COLS[idx]:<45} {importances[idx]:.4f}")

# ── Save ──────────────────────────────────────────────────────
with open(f'{OUTPUT_DIR}/model1_xgboost.pkl', 'wb') as f:
    pickle.dump(models, f)

pred_df = pd.DataFrame(all_preds, columns=[c+'_pred' for c in TARGET_COLS])
pred_df.to_csv(f'{OUTPUT_DIR}/model1_predictions.csv', index=False)

summary = {
    'model': 'XGBoost_Baseline',
    'train_time_sec': round(total_time, 2),
    'avg_MAE':  round(avg_mae, 4),
    'avg_RMSE': round(avg_rmse, 4),
    'avg_R2':   round(avg_r2, 4),
    'per_target': results,
    'top_features': [(FEAT_COLS[i], round(float(importances[i]),4))
                     for i in top_idx[:15]],
}
with open(f'{OUTPUT_DIR}/model1_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✅ Model 1 saved: model1_xgboost.pkl")
print(f"   Predictions:   model1_predictions.csv")
print(f"   Summary:       model1_summary.json")

# ── Incremental update function ───────────────────────────────
print("""
── INCREMENTAL UPDATE (for daily retraining) ──────────────
When new day's data arrives:

  import pickle, pandas as pd
  from train_model1_xgboost import FEAT_COLS, TARGET_COLS, PARAMS, CLIP_RANGES

  # Load existing models
  with open('./model_outputs/model1_xgboost.pkl', 'rb') as f:
      models = pickle.load(f)

  # Load ALL data (old + new day)
  X_all = pd.read_csv('X_train_updated.csv')[FEAT_COLS].values
  y_all = pd.read_csv('y_train_updated.csv')[TARGET_COLS].values

  # Retrain each target model
  for i, target in enumerate(TARGET_COLS):
      models[target].fit(X_all, y_all[:, i], verbose=False)

  # Save updated models
  with open('./model_outputs/model1_xgboost.pkl', 'wb') as f:
      pickle.dump(models, f)
""")

Loading data...
  X_train: (201984, 123), y_train: (201984, 4)
  X_test:  (135120, 123),  y_test:  (135120, 4)
  Features: 123

Training XGBoost models...
Target                                Train Time      MAE     RMSE       R²
---------------------------------------------------------------------------
  threshold_hours                         20.3s   0.1487   0.6541   0.1132
  threshold_inactive_min                  21.1s  33.9866  70.2625 -26.2555
  threshold_fluctuation                   10.1s   5.9487   5.9544 -520.2029
  each_hour_fluctuation                    9.6s   5.9487   5.9544 -520.2029
---------------------------------------------------------------------------
  AVERAGE                                 61.1s  11.5082  20.7064 -266.6370

Top 15 features (average importance across targets):
   1. Window_Zscore_Fluct                           0.5003
   2. Window_Inactive_Mean                          0.1669
   3. Window_Inactive_Sum                           0.1074
   4. Ni